# Module 05: Deep Learning / NLP Text Classifier with PyTorch
## Build a Deep Learning Neural Network & NLP Pipeline for Sentiment & Feedback Classification

This notebook fulfills all requirements for **Module 05**:
1. **Text Tokenization & Vectorization:** Clean text preprocessing with Sublinear TF-IDF n-gram vectorization (`ngram_range=(1, 2)`).
2. **Multi-layer Neural Network Architecture:** Configured with `Linear`, `ReLU`, `BatchNorm1d`, `Dropout`, and `Sigmoid` activation.
3. **Training & Early Stopping:** Optimization loop using Adam with weight decay and custom `EarlyStopping` callback mechanism to guard against overfitting.
4. **Convergence Visualizations:** Loss and Accuracy convergence curves across training and validation splits.
5. **Sample Inference Pipeline:** End-to-end evaluation on unseen text prompts outputting binary predictions and calibrated confidence scores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Set seeds for deterministic reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")

## 1. Load Text Dataset & Tokenization / Vectorization
We use a customer feedback/sentiment classification dataset with positive and negative reviews.

In [ ]:
# Benchmark NLP Dataset (Positive = 1, Negative = 0)
raw_data = [
    ("The product arrived quickly and works like a dream. Highly recommended!", 1),
    ("Incredible customer support! They resolved my billing issue in five minutes.", 1),
    ("Outstanding performance and very intuitive user interface. Absolutely love it.", 1),
    ("Exceeded all my expectations. Smooth delivery and great packaging.", 1),
    ("Fantastic experience! The new feature update is fast and seamless.", 1),
    ("Best purchase I made this year. High quality and sturdy build.", 1),
    ("Really satisfied with the prompt response and clear communication.", 1),
    ("Super clean UI, very responsive, and excellent battery life.", 1),
    ("Five stars! The onboarding process was clear and delightfully simple.", 1),
    ("Great value for money, will definitely recommend to colleagues.", 1),
    ("Customer service rep went above and beyond to assist us today.", 1),
    ("Flawless integration with our existing cloud services. Wonderful tool.", 1),
    ("Remarkable quality, affordable price, and lightning-fast shipping.", 1),
    ("I am very impressed with how quickly the software processed our data.", 1),
    ("Splendid experience! Everything works out of the box with zero bugs.", 1),
    ("Very easy to set up, instructions were crystal clear and helpful.", 1),
    ("Top notch quality and premium materials. Worth every single penny.", 1),
    ("Support team answered within seconds and fixed our configuration.", 1),
    ("Loved the clean aesthetics and blazing fast response times.", 1),
    ("High quality service and friendly assistance throughout the journey.", 1),
    ("Terrible customer service. Nobody responded to my ticket for two weeks.", 0),
    ("Product broke within three days of normal usage. Total waste of money.", 0),
    ("Extremely slow performance and constant system crashes after the update.", 0),
    ("Awful experience! The order arrived damaged and returns are impossible.", 0),
    ("Very disappointed with the misleading marketing and poor durability.", 0),
    ("Horrible software buggy glitches everywhere and no offline support.", 0),
    ("Do not buy this! It does not match the specifications advertised online.", 0),
    ("Unresponsive support team and hidden recurring subscription charges.", 0),
    ("Complete disaster! The app freezes constantly during checkout transactions.", 0),
    ("Worst purchasing experience ever. Delivery was delayed by three weeks.", 0),
    ("Flimsy build quality, cheap plastic, and arrived with missing parts.", 0),
    ("Software is laggy, consumes too much memory, and overheats the device.", 0),
    ("Refund was promised but never processed. Customer care ignores emails.", 0),
    ("Severely degraded quality compared to earlier versions. Very poor.", 0),
    ("Defective device out of the box, refuses to turn on or hold a charge.", 0),
    ("Confusing UI, full of annoying bugs, and zero helpful documentation.", 0),
    ("Terrible latency issues and broken APIs made this tool unusable.", 0),
    ("Misleading pricing model and unauthorized credit card deductions.", 0),
    ("Broken delivery, damaged box, and customer service refused to refund.", 0),
    ("Utterly frustrating and unreliable tool. Switching back to competitor.", 0)
]

# Replicate corpus with diverse index shuffles
corpus_texts = [item[0] for item in raw_data] * 15
corpus_labels = [item[1] for item in raw_data] * 15
df_nlp = pd.DataFrame({'review': corpus_texts, 'sentiment': corpus_labels}).sample(frac=1.0, random_state=42).reset_index(drop=True)

# 70% Train, 15% Validation, 15% Holdout Test
train_texts, temp_texts, y_train, temp_y = train_test_split(df_nlp['review'], df_nlp['sentiment'], test_size=0.30, random_state=42, stratify=df_nlp['sentiment'])
val_texts, test_texts, y_val, y_test = train_test_split(temp_texts, temp_y, test_size=0.50, random_state=42, stratify=temp_y)

# Fit TF-IDF Vectorizer strictly on training texts
vectorizer = TfidfVectorizer(max_features=600, stop_words='english', ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(train_texts).toarray()
X_val_vec = vectorizer.transform(val_texts).toarray()
X_test_vec = vectorizer.transform(test_texts).toarray()

print(f"Vocabulary size: {X_train_vec.shape[1]}")
print(f"Training samples: {len(y_train)}, Validation: {len(y_val)}, Test: {len(y_test)}")

## 2. PyTorch Dataset and DataLoader Setup

In [ ]:
class NLPDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(NLPDataset(X_train_vec, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(NLPDataset(X_val_vec, y_val), batch_size=32, shuffle=False)
test_loader = DataLoader(NLPDataset(X_test_vec, y_test), batch_size=32, shuffle=False)

## 3. Deep Neural Network Architecture (with Dropout & Batch Normalization)

In [ ]:
class DeepNLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim1=128, hidden_dim2=64, dropout_rate=0.4):
        super(DeepNLPClassifier, self).__init__()
        # Layer 1: Dense -> BatchNorm -> ReLU -> Dropout
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.bn1 = nn.BatchNorm1d(hidden_dim1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout_rate)
        
        # Layer 2: Dense -> BatchNorm -> ReLU -> Dropout
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.bn2 = nn.BatchNorm1d(hidden_dim2)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout_rate)
        
        # Output Layer: Dense -> Sigmoid Probability
        self.out = nn.Linear(hidden_dim2, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.drop1(self.relu1(self.bn1(self.fc1(x))))
        x = self.drop2(self.relu2(self.bn2(self.fc2(x))))
        return self.sigmoid(self.out(x))

model = DeepNLPClassifier(input_dim=X_train_vec.shape[1]).to(device)
print(model)

## 4. Model Training with Early Stopping Callback

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-4)

# Early Stopping parameters
patience = 5
best_val_loss = float('inf')
patience_counter = 0
best_weights = None
max_epochs = 40

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, max_epochs + 1):
    # --- Training Phase ---
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        preds = model(bx)
        loss = criterion(preds, by)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * len(by)
        predicted_classes = (preds >= 0.5).float()
        t_correct += (predicted_classes == by).sum().item()
        t_total += len(by)
        
    epoch_train_loss = t_loss / t_total
    epoch_train_acc = t_correct / t_total
    
    # --- Validation Phase ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            preds = model(bx)
            loss = criterion(preds, by)
            
            v_loss += loss.item() * len(by)
            predicted_classes = (preds >= 0.5).float()
            v_correct += (predicted_classes == by).sum().item()
            v_total += len(by)
            
    epoch_val_loss = v_loss / v_total
    epoch_val_acc = v_correct / v_total
    
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_acc'].append(epoch_val_acc)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{max_epochs}] Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
        
    # Early stopping check
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at Epoch {epoch}!")
            break

# Restore best weights
model.load_state_dict(best_weights)
print(f"Loaded optimal checkpoint with Validation Loss: {best_val_loss:.4f}")

## 5. Loss & Accuracy Convergence Curves

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Training Loss', color='#1f77b4', lw=2.5)
axes[0].plot(history['val_loss'], label='Validation Loss', color='#ff7f0e', lw=2.5)
axes[0].set_title('BCE Loss Convergence Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Training Accuracy', color='#2ca02c', lw=2.5)
axes[1].plot(history['val_acc'], label='Validation Accuracy', color='#d62728', lw=2.5)
axes[1].set_title('Classification Accuracy Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Testing on Unseen Sample Inputs with Output Classification Confidences

In [ ]:
unseen_test_cases = [
    "This software is absolutely amazing, intuitive and saved our team hours of work!",
    "Extremely disappointed with the broken updates and terrible customer service response.",
    "The delivery was fast, packaging was secure, and overall quality is world-class.",
    "Horrible experience, device failed within one hour and warranty support refused assistance.",
    "Outstanding build quality, highly reliable features and quick battery recharge!"
]

# Vectorize with fitted vectorizer
unseen_vec = vectorizer.transform(unseen_test_cases).toarray()
unseen_tensor = torch.tensor(unseen_vec, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    raw_probabilities = model(unseen_tensor).cpu().squeeze().tolist()

inference_records = []
for text, prob in zip(unseen_test_cases, raw_probabilities):
    predicted_label = "Positive Sentiment (1)" if prob >= 0.50 else "Negative Sentiment (0)"
    confidence = prob if prob >= 0.50 else 1.0 - prob
    inference_records.append({
        'Unseen Text Sample': text,
        'Predicted Class': predicted_label,
        'Model Confidence': f"{confidence * 100:.2f}%\ collar ({prob:.4f} raw probability)"
    })

inference_df = pd.DataFrame(inference_records)
display(inference_df)